### **Imports**

In [ ]:
import numpy as np
import joblib
from sklearn.datasets import load_iris  
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

RANDOM_STATE=42
MODEL_PATH="D:\\Github\\Artificial-Intelligence-Labs\\Iris Species Classifier\\model\\iris_knn_pipeline.joblib"


### **Load Iris Dataset and Basic Info**
Load built-in Iris data and store inputs (X) and labels (y) plus human-readable names.


In [4]:
iris=load_iris()
X=iris.data
y=iris.target
feature_names=iris.feature_names
target_names=iris.target_names

print('Feature Names: ',feature_names)
print('Target Names: ',target_names)

Feature Names:  ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Target Names:  ['setosa' 'versicolor' 'virginica']


In [11]:
iris.data.shape

(150, 4)

### **Split Data (70/15/15)**
Make Train/Validation/Test splits while keeping class proportions similar using stratify.
* training data 70%
* validation data 15%
* testing data 15%

In [ ]:
X_temp, X_test, y_temp, y_test = train_test_split( X,y, test_size=0.15, random_state=RANDOM_STATE, stratify=y)
val_fraction_of_temp=0.15/0.85
X_train, X_val, y_train, y_val = train_test_split( X_temp, y_temp, test_size=val_fraction_of_temp, random_state=RANDOM_STATE, stratify=y_temp)

print('\nSPlit Size: ')
print('Train:',X_train.shape[0],'Val:',X_val.shape[0],'Test:', X_test.shape[0])



SPlit Size: 
Train: 104 Val: 23 Test: 23


### **Hyperparameter tuning using validation set (finding best k)**

* finding best k value candidate k values [3,5,7,9,11]
*  Why tune k ?
    * Small k -> low bias, high variance (overfitting risk)
    * Large k -> high bias, low variance (underfitting risk)
Goal : find best balance using validation set.

* Tracking best model
```python
best_k = None
best_val_acc = -1.0
results = []
```
Variables used to :

    * Store best k 
    * Store best validation accuracy
    * Keep all results for printing

* Training loop runs model training 5 times, once for each K.
* Pipeline Creation
    `'scaler',StandardScaler()`

    Why Scaling?
    * KNN is distance-based:
        * Uses Euclidean distance
        * Larger feature scales dominate distance
    * Different ranges must standardize
    * So scaling ensures all features contribute equally.

* Model Training `pipe.fit(X_train, y_train)`

Pipepline automatically:
1. Scales training data 
2. Fits KNN model

* Validationn Prediction
    * Predict on validatoin set
    * COmpute accuracy
    * Used for hyperparameter selection

* Storing results
* Best K selection

    

In [ ]:
k_list=[3,5,7,9,11] # K Candidate
best_k=None
best_val_acc=-1.0
results=[]

# Main Loop
for k in k_list:
    # Pipeline Creation
    pipe=Pipeline(steps=[("scaler",StandardScaler()),
        ("knn",KNeighborsClassifier(n_neighbors=k))])
    # Model Training
    pipe.fit(X_train, y_train)
    # Validation Prediciton
    y_val_pred=pipe.predict(X_val)
    val_acc=accuracy_score(y_val,y_val_pred)

    # Storing Rresults
    results.append((k,val_acc))
    # Best K Seleciton
    if val_acc > best_val_acc:
        best_val_acc=val_acc
        best_k=k

# Printnig Resutls
print('\nValidation Accurates: ')
for k, acc in results:
    print(f'k={k:>2} val_accuracy={acc:.4f}')

print(f'\nBest k form validation: {best_k} (val_accuracy={best_val_acc:.4f})')


Validation Accurates: 
k= 3 val_accuracy=1.0000
k= 5 val_accuracy=1.0000
k= 7 val_accuracy=1.0000
k= 9 val_accuracy=1.0000
k=11 val_accuracy=1.0000

Best k form validation: 3 (val_accuracy=1.0000)


##### Validation Report
*We tuned the number of neighbors (K) in KNN using a validation set. A pipeline was used to standardize features before training. All tested K values achieved 100% validation accuracy due to the well-separated nature of the Iris dataset. The model selected K=3 as it was the first to achieve the highest validation performance.*

### **Retrain using Train+Val (Final Model)**
* After choosing best_k, retrain once on Train+Validation combined (stronger final training).
* After selecting besk K, validation set is  no longer needed for tuning
    So:
        * Add validation data to training
        * Give model more datea to learn patterns
        * Improve generalization before final test evaluation
`vstack` becuase X is 2D array
`h`stack` becuase y is 1D array

* Buidling Final  Pipeline 

    This Pipeline :
    1. Scales Features
    2. Applies KNN with optimal neighbors

    Using pipeline ensures:
    * Same preprocessing during training & testing
    * NO data leakage

* Traininig Final Model

In [ ]:
# Merging Train + Val data
X_train_full=np.vstack([X_train,  X_val])
y_train_full=np.hstack([y_train, y_val])

# Building final pipeline
best_pipe=Pipeline(steps=[("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=best_k))])
# Training final model
best_pipe.fit(X_train_full, y_train_full)

,steps,"[('scaler', ...), ('knn', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,n_neighbors,3
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30


### **Evaluation**
Evaluate the final model on the untouched Test set (this is your “final score”).

In [18]:
y_test_pred=best_pipe.predict(X_test)
test_acc=accuracy_score(y_test, y_test_pred)

print("\n--- Test Evaluation ---")
print("Test accuracy:",round(test_acc, 4))

cm = confusion_matrix(y_test,y_test_pred)
print("\nConfusion Matrix (rows=true, cols=pred):\n",cm)

print("\nClassification Report:")
print(classification_report(y_test,y_test_pred,target_names=target_names,digits=4))


--- Test Evaluation ---
Test accuracy: 0.9565

Confusion Matrix (rows=true, cols=pred):
 [[7 0 0]
 [0 8 0]
 [0 1 7]]

Classification Report:
              precision    recall  f1-score   support

      setosa     1.0000    1.0000    1.0000         7
  versicolor     0.8889    1.0000    0.9412         8
   virginica     1.0000    0.8750    0.9333         8

    accuracy                         0.9565        23
   macro avg     0.9630    0.9583    0.9582        23
weighted avg     0.9614    0.9565    0.9564        23



#### **Evaluation Report**
##### Interpretation

* ≈ **95.65% accuracy**
* Out of 23 test samples → only 1 misclassified

This is expected because:
* Iris dataset is small
* Classes are well separated
* KNN works very well on it



*The final KNN model (K=3) achieved 95.65% accuracy on the test set. Setosa and Versicolor were classified perfectly, while one Virginica sample was misclassified as Versicolor due to overlapping feature characteristics. Overall precision, recall, and F1-scores indicate excellent model performance.*

### **Save Model**
Save the entire pipeline (scaler + KNN) so the API can load it later.

In [ ]:
joblib.dump(best_pipe, MODEL_PATH)

['D:\\Github\\Artificial-Intelligence-Labs\\Iris Species Classifier\\model\\iris_knn_pipeline.joblib']

### **Making Prediction** 

In [20]:
one_x = X_test[0].reshape(1, -1)
pred_class_idx = best_pipe.predict(one_x)[0]
pred_proba = best_pipe.predict_proba(one_x)[0]

print("\n--- Predict One Sample ---")
print("Input features:", dict(zip(feature_names, one_x.flatten())))
print("Predicted class:", target_names[pred_class_idx])
print("Probabilities:", {name: float(p) for name, p in zip(target_names, pred_proba)})



--- Predict One Sample ---
Input features: {'sepal length (cm)': np.float64(5.0), 'sepal width (cm)': np.float64(2.3), 'petal length (cm)': np.float64(3.3), 'petal width (cm)': np.float64(1.0)}
Predicted class: versicolor
Probabilities: {np.str_('setosa'): 0.0, np.str_('versicolor'): 1.0, np.str_('virginica'): 0.0}
